# 5주차 ① 활성함수와 기울기 소실 — 실습 1~2

**목표**: ReLU·Sigmoid·Softmax 가 각각 어떤 모양이고 어디에 쓰이는지 확인하고,
Sigmoid 를 5층 쌓았을 때 **첫 층의 기울기가 얼마나 작아지는지 직접 측정**해
"ReLU 를 쓴다"를 외우지 말고 **이유로** 갖는다.

> **실행 전 확인** — 우측 상단 커널 이름이 **`Python (dl2026)`** 인지 보세요.

> **오늘의 대전제 ★★**
> ```
>   1층:  h   = W₁x + b₁
>   2층:  out = W₂h + b₂ = W₂(W₁x + b₁) + b₂ = (W₂W₁)x + (W₂b₁ + b₂) = W'x + b'
> ```
> **선형변환을 아무리 겹쳐도 선형변환 하나입니다.** 층을 100개 쌓아도 직선 하나예요.
> 층 사이에 **비선형 함수(활성함수)** 를 끼워야 비로소 "깊다"는 말에 의미가 생깁니다.
> — 중간고사 출제 1순위입니다.

## 실습 1 — 활성함수 3종 그려 보기

In [ ]:
# 셀 1 — 활성함수의 모양
import torch
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

z = torch.linspace(-6, 6, 200)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))

ax[0].plot(z, torch.relu(z));    ax[0].set_title("ReLU  —  max(0, z)")
ax[1].plot(z, torch.sigmoid(z)); ax[1].set_title("Sigmoid  —  0~1")
ax[2].bar(range(5), torch.softmax(torch.tensor([2.0, 1.0, 0.1, -1.0, 0.5]), dim=0))
ax[2].set_title("Softmax  —  합이 1인 확률")

for a in ax[:2]:
    a.axhline(0, color="gray", lw=0.5); a.axvline(0, color="gray", lw=0.5)
plt.tight_layout(); plt.show()

| 활성함수 | 식 | 어디에 쓰나 |
|---|---|---|
| **ReLU** | `max(0, z)` | **은닉층의 기본** ★ 오늘 우리가 쓸 것 |
| **Sigmoid** | `1/(1+e⁻ᶻ)` | 이진 분류의 **출력층** (4주차에 썼다) |
| **Softmax** | `eᶻⁱ / Σeᶻʲ` | **다중 분류의 출력층** — 합이 1인 확률 |

> ReLU 는 놀랄 만큼 단순합니다 — **음수면 0, 양수면 그대로.**
> 이렇게 단순한 것이 Sigmoid 를 이긴 이유를 실습 2에서 숫자로 봅니다.

In [ ]:
# 셀 2 — Softmax 는 정말 확률인가
logits = torch.tensor([2.0, 1.0, 0.1, -1.0, 0.5])
p = torch.softmax(logits, dim=0)
print("확률 :", p)
print("합   :", p.sum())

> **관찰 포인트 ★**: 합이 정확히 **1.0** 입니다. 그래서 *"이 이미지가 티셔츠일 확률 0.7,
> 바지일 확률 0.1…"* 처럼 읽을 수 있습니다.
> 4주차의 **시그모이드는 두 부류**, 오늘의 **Softmax 는 여러 부류**용입니다.

> **체크포인트**: 세 그림이 그려지고 Softmax 합이 1이면 통과.

## 실습 2 — 기울기 소실 관찰 ★★

```
   Sigmoid                      그 기울기(미분)
    1 ┤    ___------             0.25 ┤    ／＼
      │  _/                           │   /   ＼
  0.5 ┤─●───────                      │  /     ＼
      │_/                          0  ┤_/       ＼____
      └────────── z                   └──────────────── z
                                        ↑ 양끝에서 거의 0 ★
```

역전파는 각 층의 기울기를 **곱하면서** 뒤로 갑니다.
`0.2 × 0.2 × 0.2 × 0.2 × 0.2 = 0.00032` — 층이 깊어지면 앞쪽 층에 도달할 때쯤
기울기가 **사실상 0** 이 됩니다. 직접 재 봅시다.

In [ ]:
# 셀 3 — 첫 층의 기울기는 얼마나 작아지나
import torch.nn as nn

def make_net(act):
    layers = []
    for _ in range(5):                       # 은닉층 5개
        layers += [nn.Linear(64, 64), act()]
    layers += [nn.Linear(64, 1)]
    return nn.Sequential(*layers)

torch.manual_seed(0)
x = torch.randn(32, 64)
y = torch.randn(32, 1)

for name, act in [("Sigmoid", nn.Sigmoid), ("ReLU", nn.ReLU)]:
    torch.manual_seed(0)                     # 초기값을 같게
    net = make_net(act)
    loss = ((net(x) - y) ** 2).mean()
    loss.backward()

    first = net[0].weight.grad.abs().mean().item()    # 맨 앞 층
    last  = net[-1].weight.grad.abs().mean().item()   # 맨 뒤 층
    print(f"{name:8s} | 첫 층 |grad| = {first:.3e} | 마지막 층 = {last:.3e} "
          f"| 비율 = {first/last:.1e}")

> **결과 해석 ★★**: Sigmoid 쪽은 **첫 층 기울기가 마지막 층보다 수십~수백 배 작습니다.**
> 기울기가 작다는 것은 **그 층이 거의 학습되지 않는다**는 뜻입니다.
> 앞쪽 층이 안 배우면 깊게 쌓은 의미가 없습니다 — 이것이 **기울기 소실(vanishing gradient)** 입니다.

> **핵심 메시지 ★**: ReLU 는 **양수 구간에서 기울기가 정확히 1** 입니다.
> 1을 몇 번 곱해도 1이므로 앞쪽 층까지 신호가 살아서 갑니다.
> *"단순해서 좋은 것"* 의 대표적인 예이고, **중간고사 출제 지점**입니다.

> ⚠️ **차이가 잘 안 보이면** 층을 5개에서 7개로 늘려 보세요 (`range(5)` → `range(7)`).
> 값은 층 수·초기값에 따라 달라집니다.

In [ ]:
# 셀 4 (심화) — 층을 늘리면 격차가 어떻게 벌어지나
for depth in [3, 5, 7, 9]:
    torch.manual_seed(0)
    layers = []
    for _ in range(depth):
        layers += [nn.Linear(64, 64), nn.Sigmoid()]
    layers += [nn.Linear(64, 1)]
    net = nn.Sequential(*layers)

    ((net(x) - y) ** 2).mean().backward()
    ratio = net[0].weight.grad.abs().mean() / net[-1].weight.grad.abs().mean()
    print(f"Sigmoid {depth}층 | 첫 층 / 마지막 층 기울기 비율 = {ratio.item():.2e}")

> **관찰 포인트**: 층이 깊어질수록 비율이 **급격히 작아집니다.**
> "깊게 쌓으면 좋다"가 그냥은 성립하지 않는다는 뜻이고,
> 이 문제를 우회하는 아이디어가 **ReLU**(오늘) 와 **잔차 연결**(7주차 ResNet) 입니다.

## 손실함수 — 분류에는 왜 MSE 가 아닌가

| | 쓰는 곳 | PyTorch |
|---|---|---|
| **MSE** | 회귀 — 값을 맞힌다 | `nn.MSELoss()` (4주차) |
| **BCE** | 이진 분류 — 둘 중 하나 | `nn.BCEWithLogitsLoss()` (4주차) |
| **CrossEntropy** | **다중 분류 — 여럿 중 하나** ★ | `nn.CrossEntropyLoss()` (오늘) |

```
   ✗  prob = torch.softmax(model(x), dim=1)
      loss = nn.NLLLoss()(torch.log(prob), y)

   ○  logits = model(x)                       ← 날것 그대로
      loss   = nn.CrossEntropyLoss()(logits, y)
```

> **핵심 메시지 ★ (출제 지점)**: `CrossEntropyLoss` 는 **안에서 Softmax + log 를 함께 처리**합니다.
> 4주차의 `BCEWithLogitsLoss` 와 **정확히 같은 이야기**입니다 — 나눠 계산하면
> 확률이 0에 가까울 때 `log(0)` 이 되어 `inf` 가 나오지만, 합쳐 계산하면 그 문제가 없습니다.
> **수치 안정성**.

In [ ]:
# 셀 5 — 정답은 원-핫이 아니라 "정수 인덱스"
logits = torch.tensor([[2.0, 1.0, 0.1]])     # 배치 1, 클래스 3
target = torch.tensor([0])                   # ★ 정답은 0번 클래스 (원-핫 아님)
print("loss :", nn.CrossEntropyLoss()(logits, target).item())

# 직접 계산해 같은 값이 나오는지 확인
p = torch.softmax(logits, dim=1)[0]
print("손으로 :", -torch.log(p[0]).item())

> **관찰 포인트**: 정답을 `[1,0,0]` 같은 원-핫이 아니라 **정수 `0`** 으로 줍니다.
> PyTorch 의 `CrossEntropyLoss` 는 **클래스 번호**를 받습니다. 3교시에서 그대로 쓰입니다.
> 두 값이 같다는 것은 `CrossEntropyLoss` = `-log(정답 클래스의 Softmax 확률)` 이라는 뜻입니다.

> **함정 ★**: 그래서 모델 마지막에 `nn.Softmax` 를 넣으면 **두 번 씌우는 것**이 되어 학습이 나빠집니다.
> **확률을 사람이 보고 싶을 때만** 따로 `torch.softmax` 를 씌웁니다 (3교시 실습 8).

---

### 이 노트북 체크리스트

- [ ] **활성함수가 없으면 층을 쌓아도 선형**임을 식으로 보일 수 있다 ★★
- [ ] ReLU·Sigmoid·Softmax 를 그려 보고 각각 어디에 쓰는지 안다
- [ ] Softmax 출력의 합이 1인 것을 확인했다
- [ ] Sigmoid 5층에서 **첫 층 기울기가 작아지는 것**을 숫자로 확인했다 ★
- [ ] 분류에 CrossEntropy 를 쓰는 이유를 안다
- [ ] `CrossEntropyLoss` 에 Softmax 를 붙이지 않는 이유를 안다 ★
- [ ] 정답을 **정수 인덱스**로 주는 것을 확인했다